In [32]:
import requests
import json
from datetime import datetime
from pathlib import Path

In [ ]:
OVERPASS_URL = "https://lz4.overpass-api.de/api/interpreter"

overpass_query = """
[out:json][timeout:180];

// Tìm khu vực Bình Thạnh (bỏ điều kiện admin_level để tránh bị hụt)
area["name"~"Quận Bình Thạnh|Bình Thạnh"]->.bt;

// Tìm tất cả relation/area của P.19, P.22, P.25 và Thạnh Mỹ Tây
rel(area.bt)["name"~"Phường Thạnh Mỹ Tây|Thạnh Mỹ Tây|Phường 19|Phường 22|Phường 25"];
map_to_area->.tmt;

// Lấy TOÀN BỘ các đường và hẻm (mọi loại thẻ highway, không lọc bớt)
way(area.tmt)["highway"];

// Xuất toàn bộ ways và toàn bộ nodes tạo nên chúng
out body;
>;
out skel qt;
"""

In [37]:
headers = {
    "Accept": "*/*",
    "User-Agent": "JupyterNotebook_OSM_RawData_Fetcher/1.0"
}

print("Đang tải lại toàn bộ dữ liệu thô...")
response = requests.post(
    OVERPASS_URL,
    data={"data": overpass_query.strip()},
    headers=headers
)

Đang tải lại toàn bộ dữ liệu thô...


In [39]:
if response.status_code == 200:
    raw_data = response.json()
    elements = raw_data.get("elements", [])
    
    ways = [el for el in elements if el.get("type") == "way"]
    nodes = [el for el in elements if el.get("type") == "node"]
    
    # Lưu ra file để kiểm tra số dòng
    if Path("data/raw/osm").exists():
        output_dir = Path("data/raw")
    elif Path("../data/raw").exists():
        output_dir = Path("../data/raw")
    else:
        output_dir = Path(r"data\data\raw")
    if not output_dir.exists():
        raise FileNotFoundError(f"Không tìm thấy thư mục có sẵn tại: {output_dir.resolve()}")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = output_dir / f"thanh_my_tay_full_{timestamp}.json"
    
    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(raw_data, file, ensure_ascii=False, indent=2)
        
    # Đếm cả số dòng văn bản trong file JSON
    with open(file_path, "r", encoding="utf-8") as file:
        line_count = sum(1 for _ in file)

    print("==========================================")
    print(f"1. TỔNG SỐ PHẦN TỬ (elements) : {len(elements):,} phần tử")
    print(f"   - Số điểm tọa độ (nodes)   : {len(nodes):,} điểm")
    print(f"   - Số đoạn đường/hẻm (ways) : {len(ways):,} đoạn")
    print(f"2. TỔNG SỐ DÒNG TRONG FILE JSON: {line_count:,} dòng")
    print("==========================================")
else:
    print(f"Lỗi {response.status_code}: {response.text}")

1. TỔNG SỐ PHẦN TỬ (elements) : 8,520 phần tử
   - Số điểm tọa độ (nodes)   : 6,845 điểm
   - Số đoạn đường/hẻm (ways) : 1,675 đoạn
2. TỔNG SỐ DÒNG TRONG FILE JSON: 69,476 dòng
